# Narrative Mirror — Pilot 1: method checks on the pomelo story

**What this is:** exploratory experiments on one memoir passage (the pomelo / Taqwa story, condensed to 11 sentences). The question: do model representations track how the story's meaning builds up, or is apparent structure an artifact of representation geometry?

**Models:** GPT-2 (layers 2, 5, 8, 11), DistilBERT, and the sentence encoders all-mpnet-base-v2 and all-MiniLM-L6-v2.

**Human input:** phase labels and an arc of how much meaning has "landed" by each sentence, both assigned by the author (no reader data).

**How to run:** Google Colab, CPU is enough. Each cell is self-contained and reloads its model. Cell 6 pins `numpy<2`; restart the runtime after it, then run cells 7–9.

**Key results** (details and limitations in `results.md`):
- Raw cosine similarity between GPT-2 sentence states is ~0.99–1.00 for everything: geometry, not meaning.
- After centering against a baseline of unrelated sentences: z = 1.61 (sentence vectors) and z = 2.55 (prefix-conditioned states), below the pre-set z > 3 threshold, so **inconclusive**.
- Lexical control (Islamic terms vs neutral wording in the final sentence): Δ = 0.001, no effect.
- Sentence encoders separate the phases of an external-events control story more than the memoir's internal moral turn.
- Human vs model arc (all-mpnet-base-v2): Pearson r = 0.646, Spearman ρ = 0.752, n = 11. Both arcs are fixed near 0 at the start and 1 at the end by construction, which inflates the correlation.

Exploratory work, single runs, n = 1 story.


## 1. GPT-2 hidden states by layer
Sentence-level states at layers 2, 5, 8 and 11, with the author's phase labels; anchor (to S1) and consecutive similarity.


In [ ]:
# ─────────────────────────────────────────────────────────────
# NARRATIVE MIRROR — Pomelo / Taqwa Story
# Run this in Google Colab (no GPU needed)
#
# Cell 1 — install:
#   !pip install transformers torch plotly scikit-learn
#
# Cell 2 — paste everything below
# ─────────────────────────────────────────────────────────────

import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
import json

from transformers import GPT2Model, GPT2Tokenizer

# ── LOAD MODEL ───────────────────────────────────────────────
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model     = GPT2Model.from_pretrained("gpt2", output_hidden_states=True)
model.eval()

# GPT-2 has 12 layers (0–11). We extract from 4 checkpoints.
LAYERS = [2, 5, 8, 11]
print(f"GPT-2 loaded. Extracting layers: {LAYERS}")


# ── THE REAL STORY ───────────────────────────────────────────
# Every sentence is exactly as written. Nothing invented.

sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a private garden.",
    "In that garden stood a pomelo tree, its fruits large and hanging low.",
    "Months later, when my stash was almost gone, I passed by alone.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me. I realized not everything is free.",
    "But more important, I realized I'm always watched. I lost coins but I gained compass.",
]

# ── HUMAN PHASE ANNOTATION ───────────────────────────────────
# Your ground truth. Each sentence gets its own honest label.
# 5 phases: desire · temptation · transgression · accountability · transcendence

phases = [
    "desire",          # S1:  wanting sweets — pure appetite
    "desire",          # S2:  Eid payday, coins — still in desire
    "temptation",      # S3:  the shortcut, the private garden appears
    "temptation",      # S4:  the pomelo tree, fruit hanging low — the object of temptation
    "temptation",      # S5:  alone, stash almost gone — vulnerability to temptation
    "transgression",   # S6:  hand reaches out, closes around the fruit — the act
    "accountability",  # S7:  the voice, she saw everything — being caught
    "accountability",  # S8:  open your hand — submission to accountability
    "accountability",  # S9:  coins taken — consequence
    "recognition",     # S10: something shifted, not everything is free — understanding arrives
    "transcendence",   # S11: always watched, compass gained — the meaning that was always there
]

PHASE_COLORS = {
    "desire":         "#5B8FD4",
    "temptation":     "#E8A838",
    "transgression":  "#E07A5F",
    "accountability": "#7DB8A4",
    "recognition":    "#9B7FC7",
    "transcendence":  "#C8973A",
}

print(f"\nStory loaded: {len(sentences)} sentences")
print(f"Phases: {list(dict.fromkeys(phases))}")  # unique phases in order


# ── EXTRACT HIDDEN STATES ────────────────────────────────────
# For each sentence: run through GPT-2, take the last token's
# hidden state at each of our 4 target layers.
# Result: one 768-dimensional vector per layer per sentence.

def get_hidden_states(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # outputs.hidden_states: tuple of 13 tensors (layer 0 = embedding + 12 transformer layers)
    # each tensor shape: (1, seq_len, 768)
    # we take the last token position as sentence representation
    return {
        layer: outputs.hidden_states[layer][0, -1, :].numpy()
        for layer in LAYERS
    }

print("\nExtracting hidden states...")
all_states = []
for i, s in enumerate(sentences):
    states = get_hidden_states(s)
    all_states.append(states)
    print(f"  S{i+1:02d} [{phases[i]:>14}]  {s[:55]}...")

print("Done.")


# ── COSINE SIMILARITY ────────────────────────────────────────
# Two measurements:
#
# 1. CONSECUTIVE similarity: how similar is each sentence to the one before?
#    This shows where the model notices a shift in content.
#    Key question: does the model drop at S6 (transgression) and S10 (recognition)?
#
# 2. ANCHOR similarity: how similar is each sentence to S1?
#    This shows how far the model tracks the arc from beginning to end.
#    Key question: is S11 (transcendence) far from S1 (desire)?

def cosine(a, b):
    return cosine_similarity(a.reshape(1,-1), b.reshape(1,-1))[0][0]

# Consecutive: S1→S2, S2→S3, ..., S10→S11
consecutive = {
    layer: [cosine(all_states[i][layer], all_states[i+1][layer])
            for i in range(len(sentences)-1)]
    for layer in LAYERS
}

# Anchor: similarity of each sentence to S1
anchor = {
    layer: [cosine(all_states[0][layer], all_states[i][layer])
            for i in range(len(sentences))]
    for layer in LAYERS
}

print("\nConsecutive similarity (layer 11, final layer):")
transitions = [f"S{i+1}→S{i+2}" for i in range(len(sentences)-1)]
for t, v in zip(transitions, consecutive[11]):
    bar = "█" * int(v * 30)
    print(f"  {t}  {v:.3f}  {bar}")

print("\nAnchor similarity to S1 (layer 11):")
for i, v in enumerate(anchor[11]):
    bar = "█" * int(v * 30)
    marker = " ← TAQWA" if i == 10 else ""
    print(f"  S{i+1:02d} [{phases[i]:>14}]  {v:.3f}  {bar}{marker}")


# ── CHART 1: CONSECUTIVE TRAJECTORY ─────────────────────────
# Shows where the model notices semantic shifts.
# The arc should drop at S5→S6 (transgression) and S9→S10 (recognition).
# If it doesn't — that is the finding.

layer_colors = {2: "#5B8FD4", 5: "#7DB8A4", 8: "#C8973A", 11: "#C07AB8"}
layer_names  = {2: "Layer 2 (early)", 5: "Layer 5 (mid)", 8: "Layer 8 (late)", 11: "Layer 11 (final)"}

fig_traj = go.Figure()

for layer in LAYERS:
    fig_traj.add_trace(go.Scatter(
        x=transitions,
        y=consecutive[layer],
        mode="lines+markers",
        name=layer_names[layer],
        line=dict(color=layer_colors[layer], width=2),
        marker=dict(size=7),
    ))

# Mark the two key transitions where meaning shifts
for x_mark, label in [("S5→S6", "transgression"), ("S9→S10", "recognition")]:
    fig_traj.add_vline(
        x=x_mark,
        line=dict(color="rgba(200,151,58,0.4)", dash="dot", width=1.5),
        annotation_text=label,
        annotation_font_color="#C8973A",
        annotation_font_size=10,
    )

fig_traj.update_layout(
    title="<b>Meaning Trajectory — Pomelo / Taqwa Story</b><br>"
          "<sup>Cosine similarity between consecutive sentences · "
          "drops = model noticed a shift</sup>",
    xaxis_title="Sentence transition",
    yaxis_title="Cosine similarity",
    yaxis=dict(range=[0.4, 1.0]),
    template="plotly_white",
    font=dict(family="Georgia"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=420,
)

fig_traj.write_html("trajectory_pomelo.html")
print("\nSaved: trajectory_pomelo.html")


# ── CHART 2: HEATMAP ─────────────────────────────────────────
# Grid: rows = layers, columns = sentences
# Colour = similarity to S1 (desire)
# If the arc is tracked correctly:
#   - S6 (transgression) should be noticeably darker than S3–S5
#   - S11 (transcendence) should be the darkest cell in the row
# If transcendence looks similar to desire at all layers — that is the finding.

import plotly.express as px

heatmap_z = [[anchor[layer][i] for i in range(len(sentences))] for layer in LAYERS]
col_labels = [f"S{i+1}<br>{phases[i]}" for i in range(len(sentences))]
row_labels  = [layer_names[l] for l in LAYERS]

fig_heat = go.Figure(data=go.Heatmap(
    z=heatmap_z,
    x=col_labels,
    y=row_labels,
    colorscale=[
        [0.0, "#1A1410"],
        [0.4, "#8B6914"],
        [1.0, "#F0EAE0"],
    ],
    colorbar=dict(title="Similarity<br>to S1"),
    zmin=0.4,
    zmax=1.0,
    text=[[f"{v:.2f}" for v in row] for row in heatmap_z],
    texttemplate="%{text}",
    textfont=dict(size=9),
))

fig_heat.update_layout(
    title="<b>Layer × Sentence Heatmap — Pomelo / Taqwa Story</b><br>"
          "<sup>Similarity of each sentence to S1 (desire) · "
          "dark = far from opening · pale = similar to opening</sup>",
    xaxis=dict(tickangle=-30, tickfont=dict(size=10)),
    template="plotly_white",
    font=dict(family="Georgia"),
    height=320,
)

fig_heat.write_html("heatmap_pomelo.html")
print("Saved: heatmap_pomelo.html")


# ── SAVE DATA ────────────────────────────────────────────────
output = {
    "story": "pomelo_taqwa",
    "sentences": sentences,
    "phases": phases,
    "layers": LAYERS,
    "consecutive_similarity": {str(k): [float(v) for v in vals]
                                for k, vals in consecutive.items()},
    "anchor_similarity":      {str(k): [float(v) for v in vals]
                                for k, vals in anchor.items()},
}
with open("pomelo_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("Saved: pomelo_results.json")


# ── WHAT TO LOOK FOR ─────────────────────────────────────────
print("""
═══════════════════════════════════════════════════════════
WHAT TO LOOK FOR IN YOUR CHARTS
═══════════════════════════════════════════════════════════

TRAJECTORY CHART — the two dotted lines mark:
  S5→S6  the moment the hand reaches for the fruit (transgression)
  S9→S10 the moment something shifts inside (recognition)

  Does the similarity DROP at those transitions?
  → Yes at layer 11 but not layer 2 = model tracks meaning at depth
  → Flat across all layers = model missed both shifts entirely
  → Drops somewhere else = model is tracking something, but not your arc

HEATMAP — look at S11 (transcendence / Taqwa):
  Is it DARK (far from S1) or PALE (similar to S1)?
  → Dark at layer 11 = model represents Taqwa differently from desire
  → Pale at all layers = model treats the spiritual arrival as
    semantically similar to wanting sweets
    THAT IS THE FINDING.

SHARE YOUR RESULTS:
  Paste the numbers from the console output here and we go
  from there. The numbers tell us what the project actually is.
═══════════════════════════════════════════════════════════
""")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 loaded. Extracting layers: [2, 5, 8, 11]

Story loaded: 11 sentences
Phases: ['desire', 'temptation', 'transgression', 'accountability', 'recognition', 'transcendence']

Extracting hidden states...
  S01 [        desire]  As a child, I wanted money for one thing only: sweets....
  S02 [        desire]  Eid was my payday. We roamed the village from house to ...
  S03 [    temptation]  There was a secret shortcut to the candy store through ...
  S04 [    temptation]  In that garden stood a pomelo tree, its fruits large an...
  S05 [    temptation]  Months later, when my stash was almost gone, I passed b...
  S06 [ transgression]  I reached out and my fingers closed around one of the f...
  S07 [accountability]  A woman's voice stopped me. She had seen everything....
  S08 [accountability]  She asked me to open my hand. I did....
  S09 [accountability]  She took the last coins I had and walked away....
  S10 [   recognition]  In that moment, something shifted in me. I realized not..

TypeError: unsupported operand type(s) for +: 'int' and 'str'

## 2. Decisive test v0 (raw cosine)
Anchor and consecutive similarity at the last layer, a random baseline, and lexical-control variants of the final sentence.


In [ ]:
# Narrative Mirror — Decisive Test v0
# - computes anchor similarity (S1 to all)
# - computes consecutive similarity
# - builds a random baseline distribution
# - optional lexical-control variants for S11

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# 0) CONFIG
# -----------------------------
MODEL_NAME = "gpt2"          # start small; later you can try "mistralai/Mistral-7B-Instruct-v0.2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Your 11 sentences (replace with your exact ones if needed)
sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a neighbor’s garden.",
    "In that garden stood a pomelo tree, its fruits large and glowing.",
    "Months later, when my stash was almost gone, I passed by again.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me.",
    "But more important, I realized I'm always watched."
]

# Lexical-control variants for S11 (to test "vocab cueing" vs accumulation)
S11_ORIG = sentences[10]
S11_NEUTRAL = "But more important, I realized something important."
S11_ISLAMIC_CUE = "But more important, I realized Allah is always watching me."

torch.set_grad_enabled(False)

# -----------------------------
# 1) LOAD MODEL
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
model.to(DEVICE)
model.eval()
model.config.output_hidden_states = True

# GPT-2 has no pad token by default; set for batching convenience
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# 2) HELPERS
# -----------------------------
def get_last_token_vec(text: str, layer: int = -1) -> torch.Tensor:
    """
    Returns the last-token hidden state vector for `text` at a given layer.
    layer = -1 means final layer.
    """
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}

    out = model(**enc, output_hidden_states=True)
    hs = out.hidden_states[layer]          # (1, seq_len, hidden_dim)
    vec = hs[0, -1, :]                     # last token
    vec = vec / (vec.norm(p=2) + 1e-12)    # normalize for cosine sim
    return vec.detach().float().cpu()

def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.sum(a * b).item())

def batch_vectors(texts, layer=-1):
    return [get_last_token_vec(t, layer=layer) for t in texts]

# -----------------------------
# 3) CORE METRICS
# -----------------------------
def compute_story_metrics(story_texts, layer=-1):
    vecs = batch_vectors(story_texts, layer=layer)

    # consecutive
    cons = [cosine(vecs[i], vecs[i+1]) for i in range(len(vecs)-1)]

    # anchor to S1
    anchor = [cosine(vecs[0], vecs[i]) for i in range(len(vecs))]

    return vecs, cons, anchor

# -----------------------------
# 4) RANDOM BASELINE (CRITICAL)
# -----------------------------
# Create "random" sentences by shuffling words inside your story
# (cheap baseline; later upgrade to real unrelated text samples)
rng = np.random.default_rng(0)

def word_shuffle(text: str) -> str:
    words = text.split()
    if len(words) < 4:
        return text
    rng.shuffle(words)
    return " ".join(words)

random_texts = [word_shuffle(s) for s in sentences] + [
    "The solar system contains planets, moons, asteroids, and comets.",
    "In 2012, researchers proposed several methods for evaluating language models.",
    "A recipe may include flour, sugar, butter, and eggs, mixed and baked."
]

# -----------------------------
# 5) RUN DECISIVE TESTS
# -----------------------------
LAYER = -1  # final layer; later you can loop over layers

vecs, cons, anchor = compute_story_metrics(sentences, layer=LAYER)

s1 = vecs[0]
s11 = vecs[10]
sim_s1_s11 = cosine(s1, s11)

# Baseline: S1 vs random/unrelated
rand_vecs = batch_vectors(random_texts, layer=LAYER)
rand_sims = [cosine(s1, rv) for rv in rand_vecs]

# Lexical-control on S11
v_orig = get_last_token_vec(S11_ORIG, layer=LAYER)
v_neut = get_last_token_vec(S11_NEUTRAL, layer=LAYER)
v_islam = get_last_token_vec(S11_ISLAMIC_CUE, layer=LAYER)

sim_s1_orig = cosine(s1, v_orig)
sim_s1_neut = cosine(s1, v_neut)
sim_s1_islam = cosine(s1, v_islam)

# -----------------------------
# 6) PRINT RESULTS
# -----------------------------
def bar(x, width=30):
    n = int(max(0, min(1, x)) * width)
    return "█" * n + " " * (width - n)

print(f"\nModel: {MODEL_NAME} | Device: {DEVICE} | Layer: {LAYER}\n")

print("Consecutive similarity:")
for i, v in enumerate(cons, start=1):
    print(f"  S{i:02d}→S{i+1:02d}  {v:.3f}  {bar(v)}")

print("\nAnchor similarity to S1:")
for i, v in enumerate(anchor, start=1):
    tag = " ← S11" if i == 11 else ""
    print(f"  S{i:02d}  {v:.3f}  {bar(v)}{tag}")

print("\nDecisive check:")
print(f"  sim(S1, S11) = {sim_s1_s11:.3f}  {bar(sim_s1_s11)}")

print("\nRandom/unrelated baseline (S1 vs random texts):")
print(f"  mean={np.mean(rand_sims):.3f}  std={np.std(rand_sims):.3f}  min={np.min(rand_sims):.3f}  max={np.max(rand_sims):.3f}")

# A simple z-score to see how extreme sim(S1,S11) is relative to baseline
z = (sim_s1_s11 - np.mean(rand_sims)) / (np.std(rand_sims) + 1e-12)
print(f"  z-score of sim(S1,S11) vs baseline: {z:.2f}")

print("\nLexical-control variants (all compared to S1):")
print(f"  Original S11:      {sim_s1_orig:.3f}  {bar(sim_s1_orig)}")
print(f"  Neutralized S11:   {sim_s1_neut:.3f}  {bar(sim_s1_neut)}")
print(f"  Islamic-cue S11:   {sim_s1_islam:.3f}  {bar(sim_s1_islam)}")

# -----------------------------
# 7) QUICK INTERPRETATION HEURISTIC
# -----------------------------
print("\nHeuristic interpretation:")
if sim_s1_s11 > np.mean(rand_sims) + 2*np.std(rand_sims):
    print("  ✅ S11 is much closer to S1 than random text is — story stays in same basin.")
else:
    print("  ❗ S11 is not especially close to S1 compared to baseline — model likely moved.")

if abs(sim_s1_islam - sim_s1_neut) > 0.05:
    print("  ⚠️ Strong sensitivity to Islamic cueing — could be lexical pattern-matching.")
else:
    print("  ✅ Low sensitivity to Islamic cue — suggests effect is not just vocabulary.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model: gpt2 | Device: cpu | Layer: -1

Consecutive similarity:
  S01→S02  0.999  █████████████████████████████ 
  S02→S03  0.996  █████████████████████████████ 
  S03→S04  0.995  █████████████████████████████ 
  S04→S05  0.999  █████████████████████████████ 
  S05→S06  0.998  █████████████████████████████ 
  S06→S07  0.999  █████████████████████████████ 
  S07→S08  0.997  █████████████████████████████ 
  S08→S09  0.998  █████████████████████████████ 
  S09→S10  0.995  █████████████████████████████ 
  S10→S11  0.999  █████████████████████████████ 

Anchor similarity to S1:
  S01  1.000  █████████████████████████████ 
  S02  0.999  █████████████████████████████ 
  S03  0.995  █████████████████████████████ 
  S04  0.999  █████████████████████████████ 
  S05  0.999  █████████████████████████████ 
  S06  0.998  █████████████████████████████ 
  S07  0.999  █████████████████████████████ 
  S08  0.996  █████████████████████████████ 
  S09  0.996  █████████████████████████████ 
  S10  0.999  █

## 3. Prefix-conditioned vectors + lexical control
Each sentence is read in the context of the text before it. Lexical control: Islamic-cue vs neutralized final sentence (Δ = 0.001).


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a neighbor’s garden.",
    "In that garden stood a pomelo tree, its fruits large and glowing.",
    "Months later, when my stash was almost gone, I passed by again.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me.",
    "But more important, I realized I'm always watched."
]

S11_ORIG = sentences[10]
S11_NEUTRAL = "But more important, I realized something important."
S11_ISLAMIC_CUE = "But more important, I realized Allah is always watching me."

torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
model.to(DEVICE).eval()
model.config.output_hidden_states = True

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------- helpers ----------
def normalize(v: torch.Tensor) -> torch.Tensor:
    return v / (v.norm(p=2) + 1e-12)

def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.sum(a * b).item())

def encode(text: str):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    return {k: v.to(DEVICE) for k, v in enc.items()}

def pooled_span_vector(full_text: str, span_text: str, layer: int = -1) -> torch.Tensor:
    """
    Compute a vector for `span_text` INSIDE `full_text` by:
    - tokenizing full_text
    - tokenizing span_text
    - locating the last occurrence of span token ids inside full token ids
    - mean pooling hidden states over that token span (excluding obvious punctuation-only spans)
    """
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()
    span_ids = tokenizer(span_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()

    # find last occurrence of span_ids in full_ids (simple scan)
    start = None
    for i in range(len(full_ids) - len(span_ids), -1, -1):
        if full_ids[i:i+len(span_ids)] == span_ids:
            start = i
            break
    if start is None:
        raise ValueError("Could not locate span inside full text (tokenization mismatch).")

    end = start + len(span_ids)  # exclusive

    out = model(**encode(full_text), output_hidden_states=True)
    hs = out.hidden_states[layer][0]  # (seq_len, hidden_dim)

    span_hs = hs[start:end, :]

    # Optional: drop tokens that are *only* punctuation-ish to avoid period dominance.
    # We do this by decoding each token and filtering those that are mostly non-alnum.
    keep = []
    for j, tid in enumerate(full_ids[start:end]):
        tok = tokenizer.decode([tid])
        # keep token if it contains any alphanumeric character
        if any(ch.isalnum() for ch in tok):
            keep.append(j)

    if len(keep) >= 1:
        span_hs = span_hs[keep, :]

    vec = span_hs.mean(dim=0).detach().float().cpu()
    return normalize(vec)

def story_prefix(i: int, sents: list[str]) -> str:
    return "\n".join(sents[:i])

# ---------- core experiment ----------
def compute_prefix_vectors(sents: list[str], layer: int = -1):
    """
    For each i (1..N), embed prefix S1..Si, then extract a vector
    representing sentence Si within that prefix.
    """
    vecs = []
    for i in range(1, len(sents) + 1):
        prefix = story_prefix(i, sents)
        span = sents[i-1]
        vecs.append(pooled_span_vector(prefix, span, layer=layer))
    return vecs

def run_report(vecs):
    cons = [cosine(vecs[i], vecs[i+1]) for i in range(len(vecs)-1)]
    anchor = [cosine(vecs[0], vecs[i]) for i in range(len(vecs))]

    print("\nConsecutive similarity (prefix-conditioned, pooled):")
    for i, v in enumerate(cons, start=1):
        print(f"  S{i:02d}→S{i+1:02d}  {v:.3f}")

    print("\nAnchor similarity to S1:")
    for i, v in enumerate(anchor, start=1):
        tag = " ← S11" if i == 11 else ""
        print(f"  S{i:02d}  {v:.3f}{tag}")

    print(f"\nDecisive check: sim(S1,S11) = {anchor[10]:.3f}")

# run
LAYER = -1
vecs = compute_prefix_vectors(sentences, layer=LAYER)
run_report(vecs)

# ---------- lexical control (prefix-conditioned) ----------
def replace_last_sentence(sents, new_last):
    s2 = list(sents)
    s2[-1] = new_last
    return s2

vecs_orig = compute_prefix_vectors(sentences, layer=LAYER)
vecs_neut = compute_prefix_vectors(replace_last_sentence(sentences, S11_NEUTRAL), layer=LAYER)
vecs_islam = compute_prefix_vectors(replace_last_sentence(sentences, S11_ISLAMIC_CUE), layer=LAYER)

s1 = vecs_orig[0]
sim_s1_orig = cosine(s1, vecs_orig[-1])
sim_s1_neut = cosine(s1, vecs_neut[-1])
sim_s1_islam = cosine(s1, vecs_islam[-1])

print("\nLexical-control variants (prefix-conditioned, pooled; all compared to S1):")
print(f"  Original S11:    {sim_s1_orig:.3f}")
print(f"  Neutralized S11: {sim_s1_neut:.3f}")
print(f"  Islamic-cue S11: {sim_s1_islam:.3f}")
print(f"  Δ(islam - neutral) = {(sim_s1_islam - sim_s1_neut):.3f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Consecutive similarity (prefix-conditioned, pooled):
  S01→S02  0.998
  S02→S03  0.998
  S03→S04  0.997
  S04→S05  0.996
  S05→S06  0.999
  S06→S07  0.999
  S07→S08  0.998
  S08→S09  0.998
  S09→S10  0.990
  S10→S11  0.994

Anchor similarity to S1:
  S01  1.000
  S02  0.998
  S03  0.998
  S04  0.995
  S05  0.999
  S06  0.998
  S07  0.999
  S08  0.998
  S09  0.997
  S10  0.995
  S11  0.998 ← S11

Decisive check: sim(S1,S11) = 0.998

Lexical-control variants (prefix-conditioned, pooled; all compared to S1):
  Original S11:    0.998
  Neutralized S11: 0.997
  Islamic-cue S11: 0.998
  Δ(islam - neutral) = 0.001


## 4. Sentence vs prefix-state vectors against an unrelated baseline
z-scores on raw cosine (1.21 and 1.55).


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# CONFIG
# -----------------------------
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
LAYER = -1  # final layer

sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a neighbor’s garden.",
    "In that garden stood a pomelo tree, its fruits large and glowing.",
    "Months later, when my stash was almost gone, I passed by again.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me.",
    "But more important, I realized I'm always watched."
]

# A small "unrelated" set (expand this list later; 30-100 is ideal)
unrelated = [
    "The solar system contains planets, moons, asteroids, and comets.",
    "In 2012, researchers proposed several methods for evaluating language models.",
    "A recipe may include flour, sugar, butter, and eggs, mixed and baked.",
    "The concert ended late, and the crowd slowly moved toward the exits.",
    "Economic indicators can fluctuate due to policy changes and market sentiment.",
    "She opened the window to let fresh air into the room.",
    "The museum displayed artifacts from several ancient civilizations.",
    "He practiced the piano daily to improve his technique.",
    "A small change in temperature can affect chemical reaction rates.",
    "The train arrived on time despite the heavy rain.",
    "Machine learning models can overfit when trained on small datasets.",
    "The cat jumped onto the table and knocked over a glass."
]

torch.set_grad_enabled(False)

# -----------------------------
# LOAD MODEL
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
model.to(DEVICE).eval()
model.config.output_hidden_states = True

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# HELPERS
# -----------------------------
def normalize(v: torch.Tensor) -> torch.Tensor:
    return v / (v.norm(p=2) + 1e-12)

def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.sum(a * b).item())

def encode(text: str):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    return {k: v.to(DEVICE) for k, v in enc.items()}

def story_prefix(i: int, sents: list[str]) -> str:
    return "\n".join(sents[:i])

def pooled_span_vector(full_text: str, span_text: str, layer: int = -1) -> torch.Tensor:
    """
    Mean-pool hidden states for the tokens of `span_text` as it appears inside `full_text`.
    Filters out punctuation-only tokens to avoid '.' domination.
    """
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()
    span_ids = tokenizer(span_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()

    # find last occurrence of span_ids in full_ids
    start = None
    for i in range(len(full_ids) - len(span_ids), -1, -1):
        if full_ids[i:i+len(span_ids)] == span_ids:
            start = i
            break
    if start is None:
        raise ValueError("Could not locate span inside full text (tokenization mismatch).")

    end = start + len(span_ids)

    out = model(**encode(full_text), output_hidden_states=True)
    hs = out.hidden_states[layer][0]  # (seq_len, hidden_dim)

    span_hs = hs[start:end, :]

    keep = []
    for j, tid in enumerate(full_ids[start:end]):
        tok = tokenizer.decode([tid])
        if any(ch.isalnum() for ch in tok):
            keep.append(j)
    if len(keep) >= 1:
        span_hs = span_hs[keep, :]

    vec = span_hs.mean(dim=0).detach().float().cpu()
    return normalize(vec)

def prefix_state_vector(full_text: str, layer: int = -1) -> torch.Tensor:
    """
    Represents the *state of the whole prefix* using mean pooling over all content tokens.
    (This is the direct 'accumulated weight' test.)
    """
    enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)
    input_ids = enc["input_ids"][0].tolist()
    out = model(**{k: v.to(DEVICE) for k, v in enc.items()}, output_hidden_states=True)
    hs = out.hidden_states[layer][0]  # (seq_len, hidden_dim)

    keep = []
    for j, tid in enumerate(input_ids):
        tok = tokenizer.decode([tid])
        if any(ch.isalnum() for ch in tok):
            keep.append(j)

    if len(keep) == 0:
        keep = list(range(hs.shape[0]))

    vec = hs[keep, :].mean(dim=0).detach().float().cpu()
    return normalize(vec)

# -----------------------------
# 1) UNRELATED BASELINE (same extractor as your story sentence-vectors)
# -----------------------------
def sentence_vector_in_isolation(sent: str, layer: int = -1) -> torch.Tensor:
    # treat the sentence itself as both full_text and span_text (so it's identical extraction style)
    return pooled_span_vector(sent, sent, layer=layer)

def baseline_report(anchor_vec: torch.Tensor, unrelated_sents: list[str], layer: int = -1):
    vecs = [sentence_vector_in_isolation(s, layer=layer) for s in unrelated_sents]
    sims = [cosine(anchor_vec, v) for v in vecs]
    sims_np = np.array(sims)
    return {
        "mean": float(sims_np.mean()),
        "std": float(sims_np.std(ddof=0)),
        "min": float(sims_np.min()),
        "max": float(sims_np.max()),
        "sims": sims
    }

# -----------------------------
# 2) STORY: PREFIX-CONDITIONED SENTENCE VECTORS (what you already did)
# -----------------------------
def compute_prefix_sentence_vectors(sents: list[str], layer: int = -1):
    vecs = []
    for i in range(1, len(sents) + 1):
        prefix = story_prefix(i, sents)
        span = sents[i-1]
        vecs.append(pooled_span_vector(prefix, span, layer=layer))
    return vecs

# -----------------------------
# 3) STORY: PREFIX-STATE VECTORS (direct accumulated-weight test)
# -----------------------------
def compute_prefix_state_vectors(sents: list[str], layer: int = -1):
    vecs = []
    for i in range(1, len(sents) + 1):
        prefix = story_prefix(i, sents)
        vecs.append(prefix_state_vector(prefix, layer=layer))
    return vecs

# -----------------------------
# RUN + REPORTS
# -----------------------------
print(f"\nModel: {MODEL_NAME} | Device: {DEVICE} | Layer: {LAYER}\n")

# A) sentence-vectors inside growing prefix (your current metric)
sv = compute_prefix_sentence_vectors(sentences, layer=LAYER)
s1_sent = sv[0]
s11_sent = sv[-1]
sim_s1s11_sent = cosine(s1_sent, s11_sent)

cons_sent = [cosine(sv[i], sv[i+1]) for i in range(len(sv)-1)]
anchor_sent = [cosine(sv[0], sv[i]) for i in range(len(sv))]

print("A) Prefix-conditioned *sentence* vectors (pooled over Si tokens):")
print(f"   sim(S1_sentence, S11_sentence) = {sim_s1s11_sent:.3f}")
print(f"   anchor range to S1_sentence: min={min(anchor_sent):.3f}, max={max(anchor_sent):.3f}")
print(f"   biggest consecutive drop: {min(cons_sent):.3f} at step {cons_sent.index(min(cons_sent))+1:02d}→{cons_sent.index(min(cons_sent))+2:02d}")

baseA = baseline_report(s1_sent, unrelated, layer=LAYER)
zA = (sim_s1s11_sent - baseA["mean"]) / (baseA["std"] + 1e-12)
print("\n   Unrelated baseline (same extractor; S1_sentence vs unrelated):")
print(f"   mean={baseA['mean']:.3f} std={baseA['std']:.3f} min={baseA['min']:.3f} max={baseA['max']:.3f}")
print(f"   z(sim(S1_sentence,S11_sentence)) = {zA:.2f}")

# B) prefix-state vectors (this is the direct "accumulated weight" test)
pv = compute_prefix_state_vectors(sentences, layer=LAYER)
p1 = pv[0]
p11 = pv[-1]
sim_p1p11 = cosine(p1, p11)

cons_pref = [cosine(pv[i], pv[i+1]) for i in range(len(pv)-1)]
anchor_pref = [cosine(pv[0], pv[i]) for i in range(len(pv))]

print("\nB) Prefix-*state* vectors (pooled over ALL content tokens in prefix):")
print(f"   sim(prefix@S1, prefix@S11) = {sim_p1p11:.3f}")
print(f"   anchor range to prefix@S1: min={min(anchor_pref):.3f}, max={max(anchor_pref):.3f}")
print(f"   biggest consecutive drop: {min(cons_pref):.3f} at step {cons_pref.index(min(cons_pref))+1:02d}→{cons_pref.index(min(cons_pref))+2:02d}")

baseB = baseline_report(p1, unrelated, layer=LAYER)  # compare prefix-state vec to unrelated sentence vecs (still informative)
zB = (sim_p1p11 - baseB["mean"]) / (baseB["std"] + 1e-12)
print("\n   Unrelated baseline (prefix@S1 vs unrelated sentences; not perfect but sanity-checks scale):")
print(f"   mean={baseB['mean']:.3f} std={baseB['std']:.3f} min={baseB['min']:.3f} max={baseB['max']:.3f}")
print(f"   z(sim(prefix@S1,prefix@S11)) = {zB:.2f}")

# Optional: print full anchor curves if you want
# print("\nAnchor to S1_sentence:", [round(x, 3) for x in anchor_sent])
# print("Anchor to prefix@S1:", [round(x, 3) for x in anchor_pref])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model: gpt2 | Device: cpu | Layer: -1

A) Prefix-conditioned *sentence* vectors (pooled over Si tokens):
   sim(S1_sentence, S11_sentence) = 0.998
   anchor range to S1_sentence: min=0.995, max=1.000
   biggest consecutive drop: 0.990 at step 09→10

   Unrelated baseline (same extractor; S1_sentence vs unrelated):
   mean=0.995 std=0.003 min=0.988 max=0.998
   z(sim(S1_sentence,S11_sentence)) = 1.21

B) Prefix-*state* vectors (pooled over ALL content tokens in prefix):
   sim(prefix@S1, prefix@S11) = 0.999
   anchor range to prefix@S1: min=0.999, max=1.000
   biggest consecutive drop: 0.999 at step 01→02

   Unrelated baseline (prefix@S1 vs unrelated sentences; not perfect but sanity-checks scale):
   mean=0.995 std=0.003 min=0.988 max=0.998
   z(sim(prefix@S1,prefix@S11)) = 1.55


## 5. Centered cosine — the corrected test
Centering removes the direction all vectors share (anisotropy). z = 1.61 (sentence) and 2.55 (prefix): below z > 3, inconclusive.


In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# CONFIG
# -----------------------------
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
LAYER = -1  # final layer

sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a neighbor’s garden.",
    "In that garden stood a pomelo tree, its fruits large and glowing.",
    "Months later, when my stash was almost gone, I passed by again.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me.",
    "But more important, I realized I'm always watched."
]

# Unrelated baseline sentences (add more if you want; 50-200 is ideal)
unrelated = [
    "The solar system contains planets, moons, asteroids, and comets.",
    "In 2012, researchers proposed several methods for evaluating language models.",
    "A recipe may include flour, sugar, butter, and eggs, mixed and baked.",
    "The concert ended late, and the crowd slowly moved toward the exits.",
    "Economic indicators can fluctuate due to policy changes and market sentiment.",
    "She opened the window to let fresh air into the room.",
    "The museum displayed artifacts from several ancient civilizations.",
    "He practiced the piano daily to improve his technique.",
    "A small change in temperature can affect chemical reaction rates.",
    "The train arrived on time despite the heavy rain.",
    "Machine learning models can overfit when trained on small datasets.",
    "The cat jumped onto the table and knocked over a glass."
]

torch.set_grad_enabled(False)

# -----------------------------
# LOAD MODEL
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE)
model.to(DEVICE).eval()
model.config.output_hidden_states = True
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# HELPERS
# -----------------------------
def normalize(v: torch.Tensor) -> torch.Tensor:
    return v / (v.norm(p=2) + 1e-12)

def encode(text: str):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    return {k: v.to(DEVICE) for k, v in enc.items()}

def story_prefix(i: int, sents: list[str]) -> str:
    return "\n".join(sents[:i])

def pooled_span_vector(full_text: str, span_text: str, layer: int = -1) -> torch.Tensor:
    """Mean-pool hidden states over tokens of span_text as it appears inside full_text; drop punctuation-only tokens."""
    full_ids = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()
    span_ids = tokenizer(span_text, return_tensors="pt", truncation=True, max_length=1024)["input_ids"][0].tolist()

    start = None
    for i in range(len(full_ids) - len(span_ids), -1, -1):
        if full_ids[i:i+len(span_ids)] == span_ids:
            start = i
            break
    if start is None:
        raise ValueError("Could not locate span inside full text (tokenization mismatch).")
    end = start + len(span_ids)

    out = model(**encode(full_text), output_hidden_states=True)
    hs = out.hidden_states[layer][0]  # (seq_len, hidden_dim)

    span_hs = hs[start:end, :]
    keep = []
    for j, tid in enumerate(full_ids[start:end]):
        tok = tokenizer.decode([tid])
        if any(ch.isalnum() for ch in tok):
            keep.append(j)
    if keep:
        span_hs = span_hs[keep, :]

    return normalize(span_hs.mean(dim=0).detach().float().cpu())

def prefix_state_vector(full_text: str, layer: int = -1) -> torch.Tensor:
    """Mean-pool hidden states over ALL content tokens in full_text (prefix-state)."""
    enc = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=1024)
    input_ids = enc["input_ids"][0].tolist()
    out = model(**{k: v.to(DEVICE) for k, v in enc.items()}, output_hidden_states=True)
    hs = out.hidden_states[layer][0]

    keep = []
    for j, tid in enumerate(input_ids):
        tok = tokenizer.decode([tid])
        if any(ch.isalnum() for ch in tok):
            keep.append(j)
    if keep:
        hs = hs[keep, :]

    return normalize(hs.mean(dim=0).detach().float().cpu())

def sentence_vector_isolation(sent: str, layer: int = -1) -> torch.Tensor:
    return pooled_span_vector(sent, sent, layer=layer)

def mean_center(vecs: list[torch.Tensor]) -> torch.Tensor:
    M = torch.stack(vecs, dim=0)           # (N, d)
    mu = M.mean(dim=0, keepdim=True)
    Mc = M - mu
    Mc = Mc / (Mc.norm(p=2, dim=1, keepdim=True) + 1e-12)
    return Mc

def report_centered(sim, base_sims):
    base = base_sims.numpy()
    mean = base.mean()
    std = base.std()
    z = (sim - mean) / (std + 1e-12)
    return mean, std, base.min(), base.max(), z

# -----------------------------
# BUILD VECTORS
# -----------------------------
# A) prefix-conditioned sentence vectors (Si inside prefix S1..Si)
story_sent_vecs = []
for i in range(1, len(sentences) + 1):
    prefix = story_prefix(i, sentences)
    story_sent_vecs.append(pooled_span_vector(prefix, sentences[i-1], layer=LAYER))

# Baseline sentence vectors (unrelated sentences, isolated)
unrel_sent_vecs = [sentence_vector_isolation(s, layer=LAYER) for s in unrelated]

# Center everything in the same space
centered_sent = mean_center(story_sent_vecs + unrel_sent_vecs)
cs_story = centered_sent[:len(story_sent_vecs)]
cs_unrel = centered_sent[len(story_sent_vecs):]

# compute centered sim + baseline distribution
sim_s1_s11 = float((cs_story[0] * cs_story[-1]).sum().item())
base_sims = torch.mv(cs_unrel, cs_story[0])  # sims vs unrelated

m, sd, mn, mx, z = report_centered(sim_s1_s11, base_sims)

print(f"\n=== ONE DECISIVE TEST (CENTERED COSINE) — {MODEL_NAME} layer {LAYER} ===")
print(f"Centered sim(S1_sentence, S11_sentence): {sim_s1_s11:.3f}")
print(f"Centered baseline (S1 vs unrelated): mean={m:.3f} std={sd:.3f} min={mn:.3f} max={mx:.3f}")
print(f"z-score of S1↔S11 vs baseline: {z:.2f}")

# B) prefix-state vectors (whole-prefix accumulation)
story_pref_vecs = [prefix_state_vector(story_prefix(i, sentences), layer=LAYER) for i in range(1, len(sentences) + 1)]
unrel_pref_vecs = [prefix_state_vector(s, layer=LAYER) for s in unrelated]

centered_pref = mean_center(story_pref_vecs + unrel_pref_vecs)
cp_story = centered_pref[:len(story_pref_vecs)]
cp_unrel = centered_pref[len(story_pref_vecs):]

sim_p1_p11 = float((cp_story[0] * cp_story[-1]).sum().item())
base_pref = torch.mv(cp_unrel, cp_story[0])
m2, sd2, mn2, mx2, z2 = report_centered(sim_p1_p11, base_pref)

print("\nCentered sim(prefix@S1, prefix@S11):", f"{sim_p1_p11:.3f}")
print(f"Centered baseline (prefix@S1 vs unrelated): mean={m2:.3f} std={sd2:.3f} min={mn2:.3f} max={mx2:.3f}")
print(f"z-score of prefix@S1↔prefix@S11 vs baseline: {z2:.2f}")

print("\nInterpretation rule of thumb:")
print("- If z > 3: S11 is unusually close to S1 (supports 'closes too early').")
print("- If z ~ 0: S11 is not special (previous high cosines were mostly geometry).")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== ONE DECISIVE TEST (CENTERED COSINE) — gpt2 layer -1 ===
Centered sim(S1_sentence, S11_sentence): 0.268
Centered baseline (S1 vs unrelated): mean=-0.246 std=0.319 min=-0.615 max=0.406
z-score of S1↔S11 vs baseline: 1.61

Centered sim(prefix@S1, prefix@S11): 0.557
Centered baseline (prefix@S1 vs unrelated): mean=-0.242 std=0.314 min=-0.628 max=0.409
z-score of prefix@S1↔prefix@S11 vs baseline: 2.55

Interpretation rule of thumb:
- If z > 3: S11 is unusually close to S1 (supports 'closes too early').
- If z ~ 0: S11 is not special (previous high cosines were mostly geometry).


## 6. Experiment 2: memoir vs control story
GPT-2 and DistilBERT on the memoir and on a control story whose shifts are external events.


In [ ]:
# ─────────────────────────────────────────────────────────────
# NARRATIVE MIRROR — Experiment 2
# Two comparisons that tell you what the GPT-2 results actually mean
#
# Install (one cell before this):
#   !pip install "numpy<2" --force-reinstall -q
#   !pip install transformers torch plotly scikit-learn -q
# Then restart kernel, then run this file.
# ─────────────────────────────────────────────────────────────

import torch
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    GPT2Model, GPT2Tokenizer,
    DistilBertModel, DistilBertTokenizer
)

# ── EXPERIMENT A: CONTROL STORY ──────────────────────────────
# A story with an obvious, external semantic shift.
# GPT-2 should show a clear drop here if it can track meaning at all.
# If it flattens this too → GPT-2 is just too small (Interpretation A).
# If it shows drops here but NOT in the pomelo story → your finding is real.

control_sentences = [
    "It was a quiet Tuesday morning and everything felt normal.",
    "I made coffee and sat by the window watching the street.",
    "My phone rang. It was the hospital.",
    "They said my father had collapsed at work.",
    "I don't remember driving there. I just arrived.",
    "He was gone before I reached his room.",
    "I stood in the corridor holding his coat.",
    "Weeks later I found myself laughing at something on television.",
    "I stopped, surprised I could still do that.",
    "Grief does not replace you. It moves in alongside you.",
]

control_phases = [
    "ordinary",      # quiet morning
    "ordinary",      # coffee, routine
    "disruption",    # phone rings
    "disruption",    # the news
    "disruption",    # driving without memory
    "loss",          # he is gone
    "loss",          # holding the coat
    "return",        # laughing again
    "return",        # surprise at returning
    "transcendence", # grief as coexistence
]

# ── EXPERIMENT B: POMELO STORY (your real data) ───────────────
pomelo_sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a private garden.",
    "In that garden stood a pomelo tree, its fruits large and hanging low.",
    "Months later, when my stash was almost gone, I passed by alone.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me. I realized not everything is free.",
    "But more important, I realized I'm always watched. I lost coins but I gained compass.",
]

pomelo_phases = [
    "desire", "desire", "temptation", "temptation", "temptation",
    "transgression", "accountability", "accountability", "accountability",
    "recognition", "transcendence",
]


# ── LOAD MODELS ───────────────────────────────────────────────
print("Loading GPT-2...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model     = GPT2Model.from_pretrained("gpt2", output_hidden_states=True)
gpt2_model.eval()

print("Loading DistilBERT...")
bert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
bert_model     = DistilBertModel.from_pretrained("distilbert-base-uncased",
                                                   output_hidden_states=True)
bert_model.eval()
print("Both models loaded.\n")


# ── EXTRACTION FUNCTIONS ──────────────────────────────────────

def gpt2_hidden(sentence, layer=11):
    """GPT-2: last token hidden state at given layer."""
    inputs = gpt2_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = gpt2_model(**inputs)
    return out.hidden_states[layer][0, -1, :].numpy()

def bert_hidden(sentence, layer=5):
    """DistilBERT: [CLS] token hidden state at given layer.
    DistilBERT has 6 layers (0-5). Layer 5 is the final transformer layer.
    [CLS] token (position 0) is the standard sentence representation."""
    inputs = bert_tokenizer(sentence, return_tensors="pt",
                            truncation=True, max_length=128)
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.hidden_states[layer][0, 0, :].numpy()

def anchor_similarities(sentences, extract_fn):
    """Similarity of each sentence to sentence 1."""
    vectors = [extract_fn(s) for s in sentences]
    anchor  = vectors[0].reshape(1, -1)
    return [cosine_similarity(anchor, v.reshape(1,-1))[0][0] for v in vectors]

def consecutive_similarities(sentences, extract_fn):
    """Similarity between each consecutive sentence pair."""
    vectors = [extract_fn(s) for s in sentences]
    return [cosine_similarity(vectors[i].reshape(1,-1),
                               vectors[i+1].reshape(1,-1))[0][0]
            for i in range(len(vectors)-1)]


# ── RUN ALL FOUR COMBINATIONS ─────────────────────────────────
print("Running extractions (takes ~1 min)...")

results = {
    "gpt2_pomelo":   {
        "anchor":      anchor_similarities(pomelo_sentences,  gpt2_hidden),
        "consecutive": consecutive_similarities(pomelo_sentences, gpt2_hidden),
        "phases": pomelo_phases, "sentences": pomelo_sentences,
        "label": "GPT-2 · Pomelo/Taqwa (YOUR MEMOIR)",
        "color": "#C8973A",
    },
    "gpt2_control":  {
        "anchor":      anchor_similarities(control_sentences,  gpt2_hidden),
        "consecutive": consecutive_similarities(control_sentences, gpt2_hidden),
        "phases": control_phases, "sentences": control_sentences,
        "label": "GPT-2 · Control story",
        "color": "#5B8FD4",
    },
    "bert_pomelo":   {
        "anchor":      anchor_similarities(pomelo_sentences,  bert_hidden),
        "consecutive": consecutive_similarities(pomelo_sentences, bert_hidden),
        "phases": pomelo_phases, "sentences": pomelo_sentences,
        "label": "DistilBERT · Pomelo/Taqwa (YOUR MEMOIR)",
        "color": "#E07A5F",
    },
    "bert_control":  {
        "anchor":      anchor_similarities(control_sentences,  bert_hidden),
        "consecutive": consecutive_similarities(control_sentences, bert_hidden),
        "phases": control_phases, "sentences": control_sentences,
        "label": "DistilBERT · Control story",
        "color": "#7DB8A4",
    },
}

print("Done.\n")


# ── PRINT RESULTS ─────────────────────────────────────────────
for key, r in results.items():
    print(f"\n{'═'*55}")
    print(f"  {r['label']}")
    print(f"{'═'*55}")
    print(f"  Anchor similarity (to S1):")
    for i, (v, phase) in enumerate(zip(r['anchor'], r['phases'])):
        bar = "█" * int(v * 25)
        print(f"    S{i+1:02d} [{phase:>14}]  {v:.3f}  {bar}")
    anchor_range = max(r['anchor']) - min(r['anchor'])
    consec_range = max(r['consecutive']) - min(r['consecutive'])
    print(f"\n  Range (anchor):      {anchor_range:.3f}  ← higher = more variation = better tracking")
    print(f"  Range (consecutive): {consec_range:.3f}  ← higher = model notices more shifts")


# ── CHART: ANCHOR SIMILARITY COMPARISON ──────────────────────
# Four lines on one chart.
# If your memoir lines are FLATTER than the control lines
# across both models → that is the finding.
# If everything is flat → GPT-2/DistilBERT are both too small.

fig = go.Figure()

for key, r in results.items():
    x = [f"S{i+1}<br>{p}" for i, p in enumerate(r['phases'])]
    fig.add_trace(go.Scatter(
        x=x,
        y=r['anchor'],
        mode="lines+markers",
        name=r['label'],
        line=dict(color=r['color'], width=2,
                  dash="dot" if "control" in key else "solid"),
        marker=dict(size=7),
    ))

fig.update_layout(
    title="<b>Anchor Similarity Comparison</b><br>"
          "<sup>All four combinations · solid = memoir · dotted = control · "
          "flatter line = model tracks less meaning</sup>",
    xaxis_title="Sentence / Phase",
    yaxis_title="Cosine similarity to S1",
    yaxis=dict(range=[0.3, 1.05]),
    template="plotly_white",
    font=dict(family="Georgia"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=480,
)

fig.write_html("comparison_anchor.html")
print("\nSaved: comparison_anchor.html")


# ── CHART: RANGE SUMMARY ──────────────────────────────────────
# One bar per combination showing the range of similarity values.
# Higher bar = model shows more variation = better meaning tracking.
# This is the simplest summary of the finding.

labels = [r['label'] for r in results.values()]
anchor_ranges = [max(r['anchor']) - min(r['anchor']) for r in results.values()]
colors = [r['color'] for r in results.values()]

fig2 = go.Figure(data=[
    go.Bar(
        x=labels,
        y=anchor_ranges,
        marker_color=colors,
        text=[f"{v:.3f}" for v in anchor_ranges],
        textposition="outside",
    )
])

fig2.update_layout(
    title="<b>Meaning Tracking Range — Model × Story</b><br>"
          "<sup>Higher = model shows more variation across the narrative arc · "
          "lower = model treats all sentences as similar</sup>",
    yaxis_title="Range of anchor similarity (max − min)",
    yaxis=dict(range=[0, max(anchor_ranges) * 1.4]),
    template="plotly_white",
    font=dict(family="Georgia"),
    xaxis=dict(tickangle=-15),
    height=420,
)

fig2.write_html("comparison_range.html")
print("Saved: comparison_range.html")


# ── WHAT TO LOOK FOR ─────────────────────────────────────────
print("""
═══════════════════════════════════════════════════════════
HOW TO READ YOUR RESULTS
═══════════════════════════════════════════════════════════

Look at the RANGE values printed above.
Four possible outcomes:

OUTCOME 1 — Everything flat (range < 0.05 everywhere)
  → Both models are too small for this task.
  → Next step: run on Mistral-7B via HuggingFace API.
  → This is still useful: it defines a baseline.

OUTCOME 2 — Control story shows variation, memoir is flat
  → THE FINDING. The model tracks explicit dramatic shifts
    (hospital call, father's death) but not the implicit
    moral arc of your memoir.
  → This is your research contribution. Write this up.

OUTCOME 3 — DistilBERT shows more variation than GPT-2
  → Bidirectional models track meaning better than
    autoregressive ones for this type of narrative.
  → Architecture matters. Include this in your paper.

OUTCOME 4 — Memoir shows MORE variation than control
  → Unexpected. Would mean the model is sensitive to
    something in your writing specifically.
  → Investigate which sentences drive the variation.

Paste the range numbers here and we interpret together.
═══════════════════════════════════════════════════════════
""")

Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading DistilBERT...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Both models loaded.

Running extractions (takes ~1 min)...
Done.


═══════════════════════════════════════════════════════
  GPT-2 · Pomelo/Taqwa (YOUR MEMOIR)
═══════════════════════════════════════════════════════
  Anchor similarity (to S1):
    S01 [        desire]  1.000  ████████████████████████
    S02 [        desire]  0.936  ███████████████████████
    S03 [    temptation]  0.933  ███████████████████████
    S04 [    temptation]  0.892  ██████████████████████
    S05 [    temptation]  0.939  ███████████████████████
    S06 [ transgression]  0.921  ███████████████████████
    S07 [accountability]  0.889  ██████████████████████
    S08 [accountability]  0.908  ██████████████████████
    S09 [accountability]  0.919  ██████████████████████
    S10 [   recognition]  0.914  ██████████████████████
    S11 [ transcendence]  0.926  ███████████████████████

  Range (anchor):      0.111  ← higher = more variation = better tracking
  Range (consecutive): 0.069  ← higher = model notices mo

## 7. Environment fix
Pins `numpy<2`. Restart the runtime after this cell.


In [ ]:
!pip install "numpy<2" --force-reinstall -q
!pip install sentence-transformers plotly scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 83.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-pytho

## 8. Sentence encoders: phase separation
all-mpnet-base-v2 and all-MiniLM-L6-v2 on the memoir and the control story.


In [ ]:
# ─────────────────────────────────────────────────────────────
# NARRATIVE MIRROR — Experiment 3
# Sentence Transformers: models trained specifically for semantic meaning
#
# Install (one cell before this):
#   !pip install "numpy<2" --force-reinstall -q
#   !pip install sentence-transformers plotly scikit-learn -q
# Restart kernel, then run this file.
#
# Why this is different from GPT-2 / DistilBERT:
#   GPT-2 and DistilBERT were trained to predict text.
#   Sentence Transformers were trained to encode MEANING —
#   specifically to make semantically similar sentences
#   cluster together in vector space.
#   If anything can track your arc, these models can.
# ─────────────────────────────────────────────────────────────

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ── TWO MODELS ───────────────────────────────────────────────
# all-mpnet-base-v2  : best general semantic model, ~420MB
# all-MiniLM-L6-v2  : smaller/faster, ~80MB, nearly as good
# We run both so you can compare

MODEL_NAMES = [
    "all-mpnet-base-v2",
    "all-MiniLM-L6-v2",
]

# ── STORIES ──────────────────────────────────────────────────

pomelo_sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a private garden.",
    "In that garden stood a pomelo tree, its fruits large and hanging low.",
    "Months later, when my stash was almost gone, I passed by alone.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me. I realized not everything is free.",
    "But more important, I realized I'm always watched. I lost coins but I gained compass.",
]
pomelo_phases = [
    "desire", "desire", "temptation", "temptation", "temptation",
    "transgression", "accountability", "accountability", "accountability",
    "recognition", "transcendence",
]

control_sentences = [
    "It was a quiet Tuesday morning and everything felt normal.",
    "I made coffee and sat by the window watching the street.",
    "My phone rang. It was the hospital.",
    "They said my father had collapsed at work.",
    "I don't remember driving there. I just arrived.",
    "He was gone before I reached his room.",
    "I stood in the corridor holding his coat.",
    "Weeks later I found myself laughing at something on television.",
    "I stopped, surprised I could still do that.",
    "Grief does not replace you. It moves in alongside you.",
]
control_phases = [
    "ordinary", "ordinary", "disruption", "disruption", "disruption",
    "loss", "loss", "return", "return", "transcendence",
]

stories = {
    "pomelo": {"sentences": pomelo_sentences, "phases": pomelo_phases, "label": "Pomelo/Taqwa (memoir)"},
    "control": {"sentences": control_sentences, "phases": control_phases, "label": "Control (grief narrative)"},
}

PHASE_ORDER = {
    # memoir arc
    "desire": 1, "temptation": 2, "transgression": 3,
    "accountability": 4, "recognition": 5, "transcendence": 6,
    # control arc
    "ordinary": 1, "disruption": 3, "loss": 4, "return": 5,
}

COLORS = {
    "desire":         "#5B8FD4",
    "temptation":     "#E8A838",
    "transgression":  "#E07A5F",
    "accountability": "#7DB8A4",
    "recognition":    "#9B7FC7",
    "transcendence":  "#C8973A",
    "ordinary":       "#AAAAAA",
    "disruption":     "#E07A5F",
    "loss":           "#5B8FD4",
    "return":         "#7DB8A4",
}


# ── RUN ──────────────────────────────────────────────────────
all_results = {}

for model_name in MODEL_NAMES:
    print(f"\nLoading {model_name}...")
    model = SentenceTransformer(model_name)

    for story_key, story in stories.items():
        key = f"{model_name}__{story_key}"
        sentences = story["sentences"]
        phases    = story["phases"]

        print(f"  Encoding {story['label']}...")
        embeddings = model.encode(sentences, convert_to_numpy=True)

        # anchor: similarity of each sentence to S1
        anchor_sims = [
            cosine_similarity(embeddings[0:1], embeddings[i:i+1])[0][0]
            for i in range(len(sentences))
        ]

        # consecutive: similarity between adjacent sentences
        consec_sims = [
            cosine_similarity(embeddings[i:i+1], embeddings[i+1:i+2])[0][0]
            for i in range(len(sentences)-1)
        ]

        # phase-to-phase: mean similarity within same phase vs across phases
        phase_labels = list(dict.fromkeys(phases))  # unique, in order
        within, across = [], []
        for i in range(len(sentences)):
            for j in range(i+1, len(sentences)):
                sim = cosine_similarity(embeddings[i:i+1], embeddings[j:j+1])[0][0]
                if phases[i] == phases[j]:
                    within.append(sim)
                else:
                    across.append(sim)

        all_results[key] = {
            "model": model_name,
            "story": story_key,
            "label": f"{model_name.split('-')[1]} · {story['label']}",
            "sentences": sentences,
            "phases": phases,
            "embeddings": embeddings,
            "anchor_sims": anchor_sims,
            "consec_sims": consec_sims,
            "range_anchor": float(max(anchor_sims) - min(anchor_sims)),
            "range_consec": float(max(consec_sims) - min(consec_sims)),
            "within_phase_mean": float(np.mean(within)) if within else 0,
            "across_phase_mean": float(np.mean(across)) if across else 0,
            "phase_separation": float(np.mean(within) - np.mean(across)) if within and across else 0,
        }


# ── PRINT RESULTS ────────────────────────────────────────────
print("\n\n" + "═"*60)
print("RESULTS SUMMARY")
print("═"*60)

for key, r in all_results.items():
    print(f"\n{r['model']} · {stories[r['story']]['label']}")
    print(f"  Range (anchor):        {r['range_anchor']:.3f}")
    print(f"  Range (consecutive):   {r['range_consec']:.3f}")
    print(f"  Within-phase mean sim: {r['within_phase_mean']:.3f}")
    print(f"  Across-phase mean sim: {r['across_phase_mean']:.3f}")
    print(f"  Phase separation:      {r['phase_separation']:.3f}  ← KEY NUMBER")
    print()
    print("  Anchor similarity per sentence:")
    for i, (v, phase) in enumerate(zip(r['anchor_sims'], r['phases'])):
        bar = "█" * int(v * 25)
        print(f"    S{i+1:02d} [{phase:>14}]  {v:.3f}  {bar}")


# ── CHART 1: ANCHOR SIMILARITY ───────────────────────────────
fig = go.Figure()

line_styles = {
    "mpnet__pomelo":    dict(color="#C8973A", width=2.5, dash="solid"),
    "mpnet__control":   dict(color="#C8973A", width=1.5, dash="dot"),
    "MiniLM__pomelo":   dict(color="#5B8FD4", width=2.5, dash="solid"),
    "MiniLM__control":  dict(color="#5B8FD4", width=1.5, dash="dot"),
}

for key, r in all_results.items():
    short_key = key.replace("all-mpnet-base-v2", "mpnet").replace("all-MiniLM-L6-v2", "MiniLM")
    style_key = short_key.replace("all-mpnet-base-v2__", "mpnet__").replace("all-MiniLM-L6-v2__", "MiniLM__")

    # build style
    if "mpnet" in key and "pomelo" in key:
        lstyle = dict(color="#C8973A", width=2.5, dash="solid")
    elif "mpnet" in key and "control" in key:
        lstyle = dict(color="#C8973A", width=1.5, dash="dot")
    elif "MiniLM" in key and "pomelo" in key:
        lstyle = dict(color="#5B8FD4", width=2.5, dash="solid")
    else:
        lstyle = dict(color="#5B8FD4", width=1.5, dash="dot")

    x_labels = [f"S{i+1} {p}" for i, p in enumerate(r['phases'])]

    fig.add_trace(go.Scatter(
        x=x_labels,
        y=r['anchor_sims'],
        mode="lines+markers",
        name=f"{r['model'].split('-')[1]} · {'memoir' if r['story']=='pomelo' else 'control'}",
        line=lstyle,
        marker=dict(size=6),
    ))

fig.update_layout(
    title="<b>Sentence Transformers — Anchor Similarity</b><br>"
          "<sup>Solid = memoir · Dotted = control · "
          "Steeper slope = model tracks the arc better</sup>",
    xaxis_title="Sentence / Phase",
    yaxis_title="Cosine similarity to S1",
    template="plotly_white",
    font=dict(family="Georgia"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=460,
    xaxis=dict(tickangle=-35, tickfont=dict(size=9)),
)
fig.write_html("st_anchor.html")
print("\nSaved: st_anchor.html")


# ── CHART 2: PHASE SEPARATION ─────────────────────────────────
# THE KEY CHART.
# Phase separation = mean similarity within same phase MINUS
#                    mean similarity across different phases.
# Higher = model groups sentences of the same phase together.
# If this is near 0 → model cannot distinguish phases at all.
# If memoir < control → model separates your phases less well.

models_short   = ["mpnet · memoir", "mpnet · control", "MiniLM · memoir", "MiniLM · control"]
phase_seps     = [r['phase_separation'] for r in all_results.values()]
bar_colors     = ["#C8973A", "#E8C87A", "#5B8FD4", "#A0C4E8"]

fig2 = go.Figure(data=[go.Bar(
    x=models_short,
    y=phase_seps,
    marker_color=bar_colors,
    text=[f"{v:.4f}" for v in phase_seps],
    textposition="outside",
)])

fig2.update_layout(
    title="<b>Phase Separation Score</b><br>"
          "<sup>Within-phase similarity minus across-phase similarity · "
          "higher = model groups phases correctly · near 0 = cannot distinguish phases</sup>",
    yaxis_title="Phase separation",
    template="plotly_white",
    font=dict(family="Georgia"),
    height=400,
    yaxis=dict(range=[0, max(phase_seps) * 1.5] if max(phase_seps) > 0 else [0, 0.1]),
)
fig2.write_html("st_phase_separation.html")
print("Saved: st_phase_separation.html")


# ── WHAT TO LOOK FOR ─────────────────────────────────────────
print("""
═══════════════════════════════════════════════════════════
HOW TO READ THESE RESULTS
═══════════════════════════════════════════════════════════

THE KEY NUMBER IS: Phase Separation

  Near 0 for everything
  → Even sentence transformers cannot distinguish your phases.
  → The phases may be too similar linguistically.
  → OR the stories are too short (11 sentences is very small).
  → Next step: probe classifier, or longer excerpts.

  Positive for control, near 0 for memoir
  → THIS IS THE FINDING.
  → The model separates explicit dramatic phases (hospital,
    death, grief) but not implicit moral phases (desire,
    transgression, transcendence).
  → Your memoir's meaning formation is invisible to the model.
  → Write this up. This is the research contribution.

  Positive for both, but memoir lower than control
  → Partial finding. Model tracks both but yours less well.
  → Quantify the gap. That gap is your contribution.

  Positive for memoir, near 0 for control
  → Unexpected. Investigate which sentences drive it.

Paste the Phase Separation numbers here.
═══════════════════════════════════════════════════════════
""")


Loading all-mpnet-base-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (19 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Encoding Pomelo/Taqwa (memoir)...
  Encoding Control (grief narrative)...


════════════════════════════════════════════════════════════
RESULTS SUMMARY
════════════════════════════════════════════════════════════

all-mpnet-base-v2 · Pomelo/Taqwa (memoir)
  Range (anchor):        0.894
  Range (consecutive):   0.358
  Within-phase mean sim: 0.317
  Across-phase mean sim: 0.214
  Phase separation:      0.103  ← KEY NUMBER

  Anchor similarity per sentence:
    S01 [        desire]  1.000  █████████████████████████
    S02 [        desire]  0.414  ██████████
    S03 [    temptation]  0.367  █████████
    S04 [    temptation]  0.106  ██
    S05 [    temptation]  0.234  █████
    S06 [ transgression]  0.211  █████
    S07 [accountability]  0.155  ███
    S08 [accountability]  0.133  ███
    S09 [accountability]  0.286  ███████
    S10 [   recognition]  0.308  ███████
    S11 [ transcendence]  0.251  ██████

all-mpnet-base-v2 · Control (grief narrative)
  Range (anchor):        0.971
  R

## 9. Retrieval-augmented encodings
Adds retrieved memoir context to each sentence's encoding and measures the effect on phase separation.


In [ ]:
# ─────────────────────────────────────────────────────────────
# NARRATIVE MIRROR — Experiment 4: RAG Context
#
# Question: does giving the model memoir context BEFORE reading
# each pomelo sentence change how it represents that sentence?
#
# Install:
#   !pip install "numpy<2" --force-reinstall -q
#   !pip install sentence-transformers plotly scikit-learn -q
# Restart kernel, then run.
# ─────────────────────────────────────────────────────────────

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-mpnet-base-v2"

# ── THE TEST STORY ────────────────────────────────────────────
pomelo_sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a private garden.",
    "In that garden stood a pomelo tree, its fruits large and hanging low.",
    "Months later, when my stash was almost gone, I passed by alone.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me. I realized not everything is free.",
    "But more important, I realized I'm always watched. I lost coins but I gained compass.",
]
pomelo_phases = [
    "desire", "desire", "temptation", "temptation", "temptation",
    "transgression", "accountability", "accountability", "accountability",
    "recognition", "transcendence",
]

# ── THE MEMOIR CONTEXT POOL ───────────────────────────────────
# These are passages from your memoir that carry thematic weight.
# The RAG experiment retrieves the most relevant ones as context
# before encoding each pomelo sentence.
# Drawn directly from your text — nothing invented.

memoir_passages = [
    # Desire / anticipation
    "The last week of Ramadan always carried a different weight in the air. You could feel the shift—like something joyous was quietly approaching.",
    "I'd take the new outfit out of the bag and run my hands over the fabric—again and again, like I was memorising its feel.",
    "Waiting became a ritual of its own. The joy of waiting. The quiet patience that something good is coming.",
    "The magic only lasted for about a week. Then the clothes weren't new anymore. Just clothes.",
    "Joy lives temporarily in the waiting, in the build-up.",

    # Childhood / village / innocence
    "In the stillness of my secluded Circassian village, where the days were echoed by the voice of the children playing.",
    "When we were twelve, we discovered the post office. Suddenly, little girls became letter writers.",
    "It wasn't about the information. It was about the anticipation. It was our way of building joy.",
    "As a young girl with an excessive thirst for reading, I would consume every book I could find.",

    # Being watched / accountability
    "I'm always watched. I lost coins but I gained compass.",
    "I realized not everything is free. But more important, I realized I'm always watched.",
    "He knows exactly when each heart needs nourishment, and in what measure.",
    "أَلَا يَعْلَمُ مَنْ خَلَقَ — Does He who created not know, while He is the Subtle, the Acquainted?",

    # Transgression / moral weight
    "I reached out and my fingers closed around one of the fruits.",
    "She asked me to open my hand. I did. She took the last coins I had and walked away.",
    "In that moment, something shifted in me.",

    # Recognition / meaning arriving late
    "That's when I think I first sensed the fleeting nature of things.",
    "I felt an overwhelming sense of peace. I'm being taken care of.",
    "Looking back, what I asked Allah on the prayer mat was: I'm in desperate need for any good you have in-store for me.",
    "I trust my Lord will guide me to what is more right than this.",

    # Transcendence / Taqwa / divine provision
    "Taqwa. I lost coins but I gained compass.",
    "He knows who will flourish in the sun, and who will bloom only in the shade.",
    "What lost the one who found Allah, what has he found, the one who lost Allah.",
    "The sweetest taste in this fleeing world is getting to know who Allah is.",
    "Allah is as you think of Him. He show up in your life the way you perceive him.",

    # Loss / absence / grief
    "Those nights in the hospital were the loneliest of my life.",
    "No matter how carefully I arranged her medications, I couldn't keep her here.",
    "For my mum, the clock wouldn't have mattered, whether it worked or not. Time has stopped for her.",
    "My mum passed away on the 27th night of Ramadan.",
    "I picked up the phone. 'Mama,' I said. 'I'll be coming home… earlier than expected.'",

    # Rizeq / provision / divine arrangement
    "Not in a million years can one take what is written on you and for you.",
    "You never know who is the carrier of your rizeq. All you have to do is show up and wait.",
    "That is how Allah creates circumstances. He knows where every creature dwells, and where its rizq.",
    "What meant for you will never miss you and what missed you was never meant for you.",

    # Oxford / perseverance
    "I failed. Not metaphorically. I actually failed the second term.",
    "I couldn't choose the easy way out without giving it one last fight.",
    "Perhaps this journey will teach me something. About Allah.",
    "By 2019, I was finishing a master's in computer science at the best university in the world.",
]


# ── LOAD MODEL ────────────────────────────────────────────────
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("Done.\n")


# ── ENCODE EVERYTHING ─────────────────────────────────────────
print("Encoding memoir passages (context pool)...")
passage_embeddings = model.encode(memoir_passages, convert_to_numpy=True)

print("Encoding pomelo sentences (no context)...")
sentence_embeddings_plain = model.encode(pomelo_sentences, convert_to_numpy=True)


# ── RAG ENCODING ──────────────────────────────────────────────
# For each pomelo sentence, retrieve the top-3 most similar
# memoir passages. Prepend them as context. Re-encode.
# This simulates what a RAG pipeline does.

TOP_K = 3

def retrieve_context(sentence_vec, passage_vecs, passages, k=TOP_K):
    """Return top-k memoir passages most similar to this sentence."""
    sims = cosine_similarity(sentence_vec.reshape(1,-1), passage_vecs)[0]
    top_indices = np.argsort(sims)[::-1][:k]
    return [passages[i] for i in top_indices], sims[top_indices]

print("Building RAG-augmented encodings...")
sentence_embeddings_rag = []
retrieved_contexts = []

for i, (sentence, vec) in enumerate(zip(pomelo_sentences, sentence_embeddings_plain)):
    contexts, sims = retrieve_context(vec, passage_embeddings, memoir_passages)
    retrieved_contexts.append(contexts)

    # Prepend context to sentence
    context_prefix = " | ".join(contexts)
    augmented = f"Context: {context_prefix} | Sentence: {sentence}"
    rag_vec = model.encode(augmented, convert_to_numpy=True)
    sentence_embeddings_rag.append(rag_vec)

sentence_embeddings_rag = np.array(sentence_embeddings_rag)
print("Done.\n")


# ── COMPUTE SIMILARITIES ──────────────────────────────────────

def anchor_sims(embeddings):
    return [cosine_similarity(embeddings[0:1], embeddings[i:i+1])[0][0]
            for i in range(len(embeddings))]

def consec_sims(embeddings):
    return [cosine_similarity(embeddings[i:i+1], embeddings[i+1:i+2])[0][0]
            for i in range(len(embeddings)-1)]

def phase_separation(embeddings, phases):
    within, across = [], []
    for i in range(len(embeddings)):
        for j in range(i+1, len(embeddings)):
            sim = cosine_similarity(embeddings[i:i+1], embeddings[j:j+1])[0][0]
            if phases[i] == phases[j]:
                within.append(sim)
            else:
                across.append(sim)
    w = float(np.mean(within)) if within else 0
    a = float(np.mean(across)) if across else 0
    return w, a, w - a

plain_anchor = anchor_sims(sentence_embeddings_plain)
rag_anchor   = anchor_sims(sentence_embeddings_rag)

plain_consec = consec_sims(sentence_embeddings_plain)
rag_consec   = consec_sims(sentence_embeddings_rag)

plain_w, plain_a, plain_sep = phase_separation(sentence_embeddings_plain, pomelo_phases)
rag_w,   rag_a,   rag_sep   = phase_separation(sentence_embeddings_rag,   pomelo_phases)


# ── PRINT RESULTS ─────────────────────────────────────────────
print("═"*60)
print("RAG EXPERIMENT RESULTS")
print("═"*60)

print(f"\n{'Metric':<30} {'Plain':>10} {'With RAG':>10} {'Change':>10}")
print("-"*60)
print(f"{'Range (anchor)':<30} {max(plain_anchor)-min(plain_anchor):>10.3f} {max(rag_anchor)-min(rag_anchor):>10.3f} {(max(rag_anchor)-min(rag_anchor))-(max(plain_anchor)-min(plain_anchor)):>+10.3f}")
print(f"{'Range (consecutive)':<30} {max(plain_consec)-min(plain_consec):>10.3f} {max(rag_consec)-min(rag_consec):>10.3f} {(max(rag_consec)-min(rag_consec))-(max(plain_consec)-min(plain_consec)):>+10.3f}")
print(f"{'Phase separation':<30} {plain_sep:>10.3f} {rag_sep:>10.3f} {rag_sep-plain_sep:>+10.3f}")
print(f"{'Within-phase mean':<30} {plain_w:>10.3f} {rag_w:>10.3f} {rag_w-plain_w:>+10.3f}")
print(f"{'Across-phase mean':<30} {plain_a:>10.3f} {rag_a:>10.3f} {rag_a-plain_a:>+10.3f}")

print("\n\nAnchor similarity — sentence by sentence:")
print(f"{'':>4} {'Phase':>14}  {'Plain':>6}  {'RAG':>6}  {'Change':>7}  Top retrieved context")
print("-"*95)
for i, (p, r, phase) in enumerate(zip(plain_anchor, rag_anchor, pomelo_phases)):
    change = r - p
    sign = "+" if change >= 0 else ""
    ctx_snippet = retrieved_contexts[i][0][:55] + "..."
    print(f"S{i+1:02d} [{phase:>14}]  {p:.3f}  {r:.3f}  {sign}{change:.3f}   {ctx_snippet}")


# ── CHART 1: ANCHOR COMPARISON ────────────────────────────────
fig = go.Figure()

x_labels = [f"S{i+1}<br>{p}" for i, p in enumerate(pomelo_phases)]

fig.add_trace(go.Scatter(
    x=x_labels, y=plain_anchor,
    mode="lines+markers", name="Plain (no context)",
    line=dict(color="#9B9B9B", width=2, dash="dot"),
    marker=dict(size=7),
))
fig.add_trace(go.Scatter(
    x=x_labels, y=rag_anchor,
    mode="lines+markers", name="With memoir RAG context",
    line=dict(color="#C8973A", width=2.5),
    marker=dict(size=8),
))

fig.update_layout(
    title="<b>RAG Experiment — Anchor Similarity</b><br>"
          "<sup>Does memoir context change how each sentence is represented? "
          "Higher = more variation = better arc tracking</sup>",
    xaxis_title="Sentence / Phase",
    yaxis_title="Cosine similarity to S1",
    template="plotly_white",
    font=dict(family="Georgia"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=440,
    xaxis=dict(tickangle=-30, tickfont=dict(size=9)),
)
fig.write_html("rag_anchor.html")
print("\nSaved: rag_anchor.html")


# ── CHART 2: PHASE SEPARATION BAR ─────────────────────────────
fig2 = go.Figure(data=[go.Bar(
    x=["Plain", "With RAG context"],
    y=[plain_sep, rag_sep],
    marker_color=["#9B9B9B", "#C8973A"],
    text=[f"{plain_sep:.4f}", f"{rag_sep:.4f}"],
    textposition="outside",
    width=0.4,
)])
fig2.update_layout(
    title="<b>Phase Separation: Plain vs RAG</b><br>"
          "<sup>Higher = model groups same-phase sentences together more reliably</sup>",
    yaxis_title="Phase separation score",
    yaxis=dict(range=[0, max(plain_sep, rag_sep) * 1.5]),
    template="plotly_white",
    font=dict(family="Georgia"),
    height=380,
)
fig2.write_html("rag_phase_separation.html")
print("Saved: rag_phase_separation.html")


# ── WHAT TO LOOK FOR ─────────────────────────────────────────
print("""
═══════════════════════════════════════════════════════════
HOW TO READ THESE RESULTS
═══════════════════════════════════════════════════════════

Look at the CHANGE column in the table above.

KEY QUESTION: Did RAG improve phase separation?

  Phase separation INCREASES with RAG (+positive change)
  → Context helps. The model groups your phases more
    correctly when it has memoir background.
  → This means: your memoir's meaning IS accessible to
    the model — it just needs the surrounding context
    to find it. Sequential reading matters.
  → Finding: implicit moral meaning requires context
    that the model cannot infer from one sentence alone.

  Phase separation UNCHANGED or DECREASES
  → Context doesn't help. The gap is not about missing
    background — it is something deeper in the structure
    of how meaning accumulates in your writing.
  → Finding: even with relevant context retrieved, the
    model cannot track the moral arc. The problem is
    not information access — it is representational.

Also look at which SENTENCES change most:
  If S11 (Taqwa/transcendence) shifts the most with RAG
  → The model can find transcendence when given context
    about what transcendence means in your memoir.
  → That is a specific, useful finding.

Paste the Change column numbers here.
═══════════════════════════════════════════════════════════
""")

Loading all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.

Encoding memoir passages (context pool)...
Encoding pomelo sentences (no context)...
Building RAG-augmented encodings...
Done.

════════════════════════════════════════════════════════════
RAG EXPERIMENT RESULTS
════════════════════════════════════════════════════════════

Metric                              Plain   With RAG     Change
------------------------------------------------------------
Range (anchor)                      0.894      0.673     -0.221
Range (consecutive)                 0.358      0.331     -0.027
Phase separation                    0.103      0.085     -0.019
Within-phase mean                   0.317      0.505     +0.188
Across-phase mean                   0.214      0.421     +0.207


Anchor similarity — sentence by sentence:
              Phase   Plain     RAG   Change  Top retrieved context
-----------------------------------------------------------------------------------------------
S01 [        desire]  1.000  1.000  +0.000   As a young girl with 

## 10. Experiment 5: human arc vs model arc
Author's cumulative meaning arc vs the model's projection onto the S1→S11 direction (all-mpnet-base-v2). Note: both arcs run from ~0 to 1 by construction.


In [ ]:
# ─────────────────────────────────────────────────────────────
# NARRATIVE MIRROR — Experiment 5: Human vs Model Arc
#
# Core methodology:
#   Human journey:  h_i ∈ [0,1]  — your annotation per sentence
#   Model journey:  m_i ∈ [0,1]  — projection onto S1→S11 direction
#   Comparison:     two curves, same scale, divergence is the finding
#
# Install:
#   !pip install "numpy<2" --force-reinstall -q
#   !pip install sentence-transformers plotly scikit-learn -q
# Restart kernel, then run.
# ─────────────────────────────────────────────────────────────

import numpy as np
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr, pearsonr
from sentence_transformers import SentenceTransformer

# ── THE STORY ─────────────────────────────────────────────────
sentences = [
    "As a child, I wanted money for one thing only: sweets.",
    "Eid was my payday. We roamed the village from house to house collecting coins.",
    "There was a secret shortcut to the candy store through a private garden.",
    "In that garden stood a pomelo tree, its fruits large and hanging low.",
    "Months later, when my stash was almost gone, I passed by alone.",
    "I reached out and my fingers closed around one of the fruits.",
    "A woman's voice stopped me. She had seen everything.",
    "She asked me to open my hand. I did.",
    "She took the last coins I had and walked away.",
    "In that moment, something shifted in me. I realized not everything is free.",
    "But more important, I realized I'm always watched. I lost coins but I gained compass.",
]

phases = [
    "desire", "desire", "temptation", "temptation", "temptation",
    "transgression", "accountability", "accountability", "accountability",
    "recognition", "transcendence",
]

# ── HUMAN JOURNEY ─────────────────────────────────────────────
# h_i ∈ [0,1]: how much of what this story is really about
# has arrived for the reader by this sentence?
#
# These are PLACEHOLDER values.
# Replace them with your own numbers before running.
# Ask yourself sentence by sentence:
# "How much of the total meaning has landed by here?"
# 0 = nothing yet   1 = full arrival
#
# The shape matters more than exact values.
# Trust your felt sense as the author.

human_arc = [
    0.05,   # S1:  child wants sweets — almost nothing yet
    0.10,   # S2:  Eid, coins — still setup
    0.10,   # S3:  shortcut, garden — something being prepared
    0.15,   # S4:  pomelo tree — object appears
    0.10,   # S5:  alone, stash almost gone — vulnerability
    0.6,   # S6:  hand reaches — the act
    0.7,   # S7:  woman's voice — the turn
    0.8,   # S8:  open your hand — surrender
    0.9,   # S9:  coins taken — consequence
    0.95,   # S10: something shifted — understanding begins
    1.00,   # S11: always watched, Taqwa — full arrival
]

# ── LOAD MODEL ────────────────────────────────────────────────
print("Loading model...")
model = SentenceTransformer("all-mpnet-base-v2")
print("Done.\n")

# ── ENCODE ───────────────────────────────────────────────────
print("Encoding sentences...")
embeddings = model.encode(sentences, convert_to_numpy=True)  # shape: (11, 768)


# ── MODEL JOURNEY ─────────────────────────────────────────────
# Step 1: define arc direction in vector space
v1  = embeddings[0]   # S1 vector
v11 = embeddings[-1]  # S11 vector
direction = v11 - v1
direction = direction / np.linalg.norm(direction)  # unit vector

# Step 2: project each sentence onto that direction
projections = np.array([
    np.dot(embeddings[i] - v1, direction)
    for i in range(len(sentences))
])

# Step 3: rescale to [0,1]
p_min = projections.min()
p_max = projections.max()
model_arc = (projections - p_min) / (p_max - p_min)

# ── PRINT RESULTS ─────────────────────────────────────────────
print("\n" + "═"*65)
print("HUMAN ARC vs MODEL ARC")
print("═"*65)
print(f"{'':>4} {'Phase':>14}  {'Human':>6}  {'Model':>6}  {'Gap':>7}  {'Direction'}")
print("-"*65)

for i, (h, m, phase) in enumerate(zip(human_arc, model_arc, phases)):
    gap = m - h
    sign = "+" if gap >= 0 else ""
    # direction of gap
    if abs(gap) < 0.05:
        direction_label = "aligned"
    elif gap > 0:
        direction_label = "model ahead of human"
    else:
        direction_label = "model behind human"
    print(f"S{i+1:02d} [{phase:>14}]  {h:.3f}  {m:.3f}  {sign}{gap:.3f}  {direction_label}")

# ── CORRELATION ───────────────────────────────────────────────
pearson_r,  pearson_p  = pearsonr(human_arc, model_arc)
spearman_r, spearman_p = spearmanr(human_arc, model_arc)

print(f"\nPearson correlation:  r = {pearson_r:.3f}  (p = {pearson_p:.3f})")
print(f"Spearman correlation: r = {spearman_r:.3f}  (p = {spearman_p:.3f})")
print()
print("Interpretation:")
if pearson_r > 0.8:
    print("  Strong correlation — model journey resembles human journey")
elif pearson_r > 0.5:
    print("  Moderate correlation — model partially tracks the arc")
elif pearson_r > 0.2:
    print("  Weak correlation — model loosely follows the arc")
else:
    print("  No correlation — model journey bears no relation to human journey")

# Maximum divergence point
gaps = [abs(m - h) for m, h in zip(model_arc, human_arc)]
max_gap_idx = np.argmax(gaps)
print(f"\nMaximum divergence at S{max_gap_idx+1} [{phases[max_gap_idx]}]")
print(f"  Human: {human_arc[max_gap_idx]:.3f}  Model: {model_arc[max_gap_idx]:.3f}  Gap: {gaps[max_gap_idx]:.3f}")
print(f"  Sentence: \"{sentences[max_gap_idx]}\"")


# ── CHART ─────────────────────────────────────────────────────
x_labels = [f"S{i+1}<br><i>{p}</i>" for i, p in enumerate(phases)]

fig = go.Figure()

# Human arc
fig.add_trace(go.Scatter(
    x=x_labels,
    y=human_arc,
    mode="lines+markers",
    name="Human arc (author annotation)",
    line=dict(color="#C8973A", width=3),
    marker=dict(size=9, symbol="circle"),
))

# Model arc
fig.add_trace(go.Scatter(
    x=x_labels,
    y=model_arc,
    mode="lines+markers",
    name="Model arc (mpnet projection)",
    line=dict(color="#5B8FD4", width=2.5, dash="dot"),
    marker=dict(size=8, symbol="diamond"),
))

# Shade the gap between the two curves
fig.add_trace(go.Scatter(
    x=x_labels + x_labels[::-1],
    y=human_arc + list(reversed(model_arc)),
    fill="toself",
    fillcolor="rgba(224, 122, 95, 0.12)",
    line=dict(color="rgba(255,255,255,0)"),
    name="Divergence gap",
    showlegend=True,
))

# Mark the maximum divergence point
fig.add_annotation(
    x=x_labels[max_gap_idx],
    y=max(human_arc[max_gap_idx], model_arc[max_gap_idx]) + 0.06,
    text=f"Max divergence<br>S{max_gap_idx+1}: {gaps[max_gap_idx]:.2f}",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#E07A5F",
    font=dict(color="#E07A5F", size=11),
    bgcolor="rgba(255,255,255,0.8)",
)

fig.update_layout(
    title="<b>Human Arc vs Model Arc — Pomelo / Taqwa Story</b><br>"
          "<sup>How much meaning has arrived at each sentence · "
          "gap = what the model missed</sup>",
    xaxis_title="Sentence",
    yaxis_title="Arc progress (0 = nothing yet · 1 = full arrival)",
    yaxis=dict(range=[-0.05, 1.15]),
    template="plotly_white",
    font=dict(family="Georgia", size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=500,
    xaxis=dict(tickangle=-20, tickfont=dict(size=10)),
)

fig.write_html("human_vs_model_arc.html")
print("\nSaved: human_vs_model_arc.html")
print("\nOpen this file in your browser.")
print("The gap between the two lines is your finding.")
print("\nIMPORTANT: Replace the human_arc values with your own")
print("numbers before drawing conclusions.")

Loading model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Done.

Encoding sentences...

═════════════════════════════════════════════════════════════════
HUMAN ARC vs MODEL ARC
═════════════════════════════════════════════════════════════════
              Phase   Human   Model      Gap  Direction
-----------------------------------------------------------------
S01 [        desire]  0.050  0.000  -0.050  model behind human
S02 [        desire]  0.100  0.440  +0.340  model ahead of human
S03 [    temptation]  0.100  0.352  +0.252  model ahead of human
S04 [    temptation]  0.150  0.450  +0.300  model ahead of human
S05 [    temptation]  0.100  0.584  +0.484  model ahead of human
S06 [ transgression]  0.600  0.436  -0.164  model behind human
S07 [accountability]  0.700  0.613  -0.087  model behind human
S08 [accountability]  0.800  0.485  -0.315  model behind human
S09 [accountability]  0.900  0.527  -0.373  model behind human
S10 [   recognition]  0.950  0.585  -0.365  model behind human
S11 [ transcendence]  1.000  1.000  +0.000  aligned

Pe